# LeetCode 701 — Insert into a Binary Search Tree bằng Python

Link đề: https://leetcode.com/problems/insert-into-a-binary-search-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: notebook tự dựng `TreeNode`, hàm `build_tree` (dựng cây từ list kiểu LeetCode như `[4,2,7,1,3]`, `None` là nút rỗng) và `tree_to_list` (xuất cây ra list). Bài này có **nhiều đáp án đúng** (đề cho phép trả về bất kỳ BST hợp lệ nào chứa thêm `val`), nên phần test dùng **hàm kiểm tra BST hợp lệ + kiểm tra đúng tập giá trị**, không so khớp từng chữ với một đáp án duy nhất (trừ khi nói rõ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho gốc của một **cây tìm kiếm nhị phân (BST)** và một giá trị `val` **chưa có** trong cây; hãy chèn `val` sao cho cây vẫn là BST rồi trả về gốc. Có thể có nhiều cách chèn hợp lệ, trả về cách nào cũng được.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, đệ quy trên cây, vòng `while` đi xuống cây với con trỏ `cur`, `collections.deque` để duyệt theo tầng, duyệt **inorder** để kiểm tra tính chất BST.
3. Đi từ **lấy inorder, chèn vào danh sách, dựng lại cả cây** → **đệ quy chèn vào lá** → **lặp chèn vào lá** → **chèn tại gốc bằng cách cắt (split) cây theo `val`**.
4. Hiểu mấu chốt: trong BST, so sánh `val` với nút hiện tại cho biết **đi trái hay phải**; đi đến chỗ trống thì **gắn nút mới vào đó**. Chỉ phải đi theo **một đường** từ gốc xuống nên tốn `O(h)` với `h` là chiều cao.
5. Tránh các lỗi kinh điển: quên xử lý cây rỗng, quên gán lại kết quả của lời gọi đệ quy (`root.left = ...`), đè mất cây con thay vì đi tiếp xuống, để con trỏ `cur` rơi ra `None` rồi mất nút cha, so sánh sai chiều, và đệ quy quá sâu trên cây lệch.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một **cây tìm kiếm nhị phân** (BST) và một giá trị `val` cần chèn vào cây. Trả về **nút gốc của BST sau khi chèn**. Đảm bảo giá trị mới **không tồn tại** trong BST gốc.

Chú ý: có thể có nhiều cách chèn hợp lệ, miễn là cây sau khi chèn **vẫn là BST**; bạn có thể trả về **bất kỳ cách nào**.

**Ví dụ 1**

- Input: `root = [4,2,7,1,3]`, `val = 5`
- Output: `[4,2,7,1,3,5]`
- Giải thích: `5 > 4` đi phải, `5 < 7` đi trái, chỗ đó trống nên `5` là con trái của `7`. Cũng có đáp án hợp lệ khác (đề minh họa thêm một cây khác); mọi BST chứa `{1,2,3,4,5,7}` đều được.

**Ví dụ 2**

- Input: `root = [40,20,60,10,30,50,70]`, `val = 25`
- Output: `[40,20,60,10,30,50,70,null,null,25]`
- Giải thích: `25 < 40` trái, `25 > 20` phải sang `30`, `25 < 30` trái → `25` là con trái của `30`.

**Ví dụ 3**

- Input: `root = [4,2,7,1,3,null,null,null,null,null,null]`, `val = 5`
- Output: `[4,2,7,1,3,5]`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = []`, `val = 5`
- Output: `[5]`
- Giải thích: cây rỗng thì nút mới trở thành gốc.

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[0, 10^4]` (**có thể bằng 0**).
- `-10^8 <= Node.val <= 10^8`; mọi `Node.val` **khác nhau**.
- `-10^8 <= val <= 10^8`; `val` **chưa có** trong BST gốc.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([4, 2, 7, 1, 3], 5, [4, 2, 7, 1, 3, 5]),
    ([40, 20, 60, 10, 30, 50, 70], 25, [40, 20, 60, 10, 30, 50, 70, None, None, 25]),
    ([4, 2, 7, 1, 3, None, None, None, None, None, None], 5, [4, 2, 7, 1, 3, 5]),
    ([], 5, [5]),
]
for vals, v, expected in examples:
    print(f"root = {vals}, val = {v} -> một đáp án: {expected}")

root = [4, 2, 7, 1, 3], val = 5 -> một đáp án: [4, 2, 7, 1, 3, 5]
root = [40, 20, 60, 10, 30, 50, 70], val = 25 -> một đáp án: [40, 20, 60, 10, 30, 50, 70, None, None, 25]
root = [4, 2, 7, 1, 3, None, None, None, None, None, None], val = 5 -> một đáp án: [4, 2, 7, 1, 3, 5]
root = [], val = 5 -> một đáp án: [5]


## 3. Kiến thức nền

### 3.1. Cây nhị phân, nút `TreeNode` và dựng cây từ list

Mỗi nút `TreeNode` có `val`, `left`, `right` (con trỏ tới nút con hoặc `None`). List kiểu LeetCode liệt kê các nút **theo từng tầng từ trái sang phải**, `None` là chỗ trống. `build_tree` dựng cây từ list, `tree_to_list` làm ngược lại (cắt các `None` thừa ở cuối).

In [2]:
root = build_tree([4, 2, 7, 1, 3])
print("gốc:", root.val, "| con trái:", root.left.val, "| con phải:", root.right.val)
print("con trái của 2:", root.left.left.val, "| con phải của 2:", root.left.right.val)
print("con của 7:", root.right.left, root.right.right)
print(tree_to_list(root))
print("cây rỗng:", build_tree([]), "->", tree_to_list(None))

gốc: 4 | con trái: 2 | con phải: 7
con trái của 2: 1 | con phải của 2: 3
con của 7: None None
[4, 2, 7, 1, 3]
cây rỗng: None -> []


Để nhìn cây trực quan, ta viết `print_tree` in cây **xoay ngang** (gốc ở bên trái, con phải ở trên, con trái ở dưới):

In [3]:
def print_tree(node, depth=0):
    if node is None:
        return
    print_tree(node.right, depth + 1)
    print("    " * depth + str(node.val))
    print_tree(node.left, depth + 1)

print_tree(build_tree([4, 2, 7, 1, 3]))

    7
4
        3
    2
        1


### 3.2. Cây tìm kiếm nhị phân (BST) và tính chất then chốt

**BST**: với **mọi** nút `x`, mọi giá trị trong cây con **trái** của `x` đều **nhỏ hơn** `x.val`, và mọi giá trị trong cây con **phải** đều **lớn hơn** `x.val` (ở bài này các giá trị khác nhau nên không có "bằng"). Hệ quả đẹp: **duyệt inorder** (trái → gốc → phải) cho ra dãy **tăng dần**.

Vì vậy, để biết `val` nên nằm ở đâu, ta so sánh với nút hiện tại: `val < x.val` thì `val` thuộc cây con trái, ngược lại thuộc cây con phải — **loại cả một nửa** cây ở mỗi bước (nếu cây cân bằng).

In [4]:
def inorder(root):
    """Duyệt inorder bằng ngăn xếp (không đệ quy, an toàn với cây sâu)."""
    out, stack, cur = [], [], root
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        out.append(cur.val)
        cur = cur.right
    return out

t = build_tree([40, 20, 60, 10, 30, 50, 70])
print("inorder:", inorder(t), "(tăng dần vì là BST)")
t2 = build_tree([4, 7, 2])           # 7 là con trái của 4: KHÔNG phải BST
print("inorder của cây không phải BST:", inorder(t2))

inorder: [10, 20, 30, 40, 50, 60, 70] (tăng dần vì là BST)
inorder của cây không phải BST: [7, 4, 2]


### 3.3. Hàm kiểm tra BST hợp lệ

Chỉ so sánh nút với **con trực tiếp** là chưa đủ (một nút ở sâu bên phải-trái vẫn phải nhỏ hơn tổ tiên). Cách đúng: mỗi nút mang theo một **khoảng cho phép** `(lo, hi)` — giá trị phải **nằm ngặt** giữa `lo` và `hi`; đi trái thì `hi` thu về `x.val`, đi phải thì `lo` nâng lên `x.val`. Ta dùng hàm này làm **bộ kiểm tra đáp án** (vì có nhiều đáp án đúng).

In [5]:
def is_valid_bst(root):
    """Kiểm tra BST hợp lệ (giá trị phải tăng ngặt theo inorder), bằng ngăn xếp kèm (lo, hi)."""
    if root is None:
        return True
    stack = [(root, float("-inf"), float("inf"))]
    while stack:
        node, lo, hi = stack.pop()
        if not (lo < node.val < hi):
            return False
        if node.left:
            stack.append((node.left, lo, node.val))
        if node.right:
            stack.append((node.right, node.val, hi))
    return True

print(is_valid_bst(build_tree([4, 2, 7, 1, 3])))
print(is_valid_bst(build_tree([4, 2, 7, 1, 3, 5])))
print(is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])))     # 3 nằm ở cây con phải của 5 nhưng 3 < 5
print(is_valid_bst(build_tree([4, 7, 2])))

True
True
False
False


### 3.4. Đệ quy trên cây

Đệ quy trên cây thường có dạng: "nếu nút rỗng thì xử lý cơ sở; ngược lại xử lý nút hiện tại rồi gọi cho con trái/con phải". Điểm quan trọng khi **thay đổi cây**: hàm **trả về nút gốc của cây con đã xử lý** và ta **gán lại** kết quả vào con trỏ của nút cha (`node.left = hàm(node.left)`).

In [6]:
def height(node):
    """Chiều cao cây (số nút trên đường dài nhất từ gốc xuống lá); cây rỗng có chiều cao 0."""
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

print(height(build_tree([])), height(build_tree([4])), height(build_tree([4, 2, 7, 1, 3])))

0 1 3


### 3.5. Vòng lặp `while` đi xuống cây với con trỏ `cur`

Thay vì đệ quy ta có thể đi bằng vòng `while`: `cur = root`; mỗi bước `cur = cur.left` hoặc `cur = cur.right`. Cái bẫy: khi `cur` thành `None` ta đã **rời khỏi cây** — muốn gắn nút mới phải nhớ **nút cha** (hoặc dừng khi con cần đi tới là `None`).

### 3.6. Giới hạn đệ quy của Python

Python mặc định chỉ cho phép khoảng `1000` tầng đệ quy. Một BST có `10^4` nút nhưng bị **lệch hẳn** về một phía (như một chuỗi) có chiều cao `10^4` ⇒ đệ quy thuần có thể gây `RecursionError`. Ta sẽ ưu tiên cách **lặp** khi nộp.

In [7]:
def chain_tree(n):
    """BST lệch: chèn 1, 2, ..., n (mỗi nút là con phải của nút trước)."""
    root = TreeNode(1)
    cur = root
    for v in range(2, n + 1):
        cur.right = TreeNode(v)
        cur = cur.right
    return root

try:
    height(chain_tree(5000))
except RecursionError:
    print("height(đệ quy) trên chuỗi 5000 nút -> RecursionError")
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

height(đệ quy) trên chuỗi 5000 nút -> RecursionError
giới hạn đệ quy mặc định: 1000


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy BST `[40,20,60,10,30,50,70]` và chèn `val = 25`:

```
          40
        /    \
      20      60
     /  \    /  \
    10  30  50  70
```

Hãy đi từ gốc xuống như khi **tra cứu** `25`:

| bước | nút hiện tại | so sánh | đi về |
|---|---|---|---|
| 1 | `40` | `25 < 40` | trái → `20` |
| 2 | `20` | `25 > 20` | phải → `30` |
| 3 | `30` | `25 < 30` | trái → **trống** |

Chỗ trống ở "con trái của `30`" chính là nơi `25` phải ở: mọi giá trị nằm giữa `20` và `30` đều thuộc khoảng này. Gắn nút `25` vào đó và **không cần động đến phần còn lại** của cây. Kết quả: `[40,20,60,10,30,50,70,null,null,25]`.

Nhận xét: đường đi này **duy nhất** (mỗi nút chỉ có một hướng đúng) và kết thúc ở một chỗ trống **luôn tồn tại**. Số bước bằng độ sâu của chỗ trống, tối đa chiều cao cây.

In [8]:
root = build_tree([40, 20, 60, 10, 30, 50, 70])
val = 25
cur, path = root, []
while True:
    path.append(cur.val)
    nxt = cur.left if val < cur.val else cur.right
    side = "trái" if val < cur.val else "phải"
    print(f"tại {cur.val}: {val} {'<' if val < cur.val else '>'} {cur.val} -> đi {side}", "(trống!)" if nxt is None else f"-> {nxt.val}")
    if nxt is None:
        break
    cur = nxt
print("đường đi:", path, "| chèn làm con", side, "của", cur.val)

tại 40: 25 < 40 -> đi trái -> 20
tại 20: 25 > 20 -> đi phải -> 30
tại 30: 25 < 30 -> đi trái (trống!)
đường đi: [40, 20, 30] | chèn làm con trái của 30


## 5. Cách 1: Brute Force (lấy inorder, chèn vào danh sách, dựng lại cả cây)

### Ý tưởng

Một cách "ngây thơ" nhưng đúng: **bỏ qua cấu trúc cây hiện có**.

1. Duyệt inorder lấy danh sách giá trị đã **sắp xếp** của BST.
2. Chèn `val` vào đúng chỗ của danh sách (dùng `bisect.insort`).
3. **Dựng lại toàn bộ BST** từ danh sách đã sắp xếp: lấy phần tử giữa làm gốc, đệ quy cho nửa trái và nửa phải (sẽ ra cây cân bằng).

Cây trả về là BST hợp lệ chứa đúng các giá trị cần có — nhưng hình dạng khác hẳn cây ban đầu (đề cho phép).

In [9]:
from bisect import insort


def build_balanced(sorted_vals, lo=0, hi=None):
    if hi is None:
        hi = len(sorted_vals) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    node = TreeNode(sorted_vals[mid])
    node.left = build_balanced(sorted_vals, lo, mid - 1)
    node.right = build_balanced(sorted_vals, mid + 1, hi)
    return node


def insert_brute(root, val):
    vals = inorder(root)         # đã tăng dần
    insort(vals, val)            # chèn giữ thứ tự
    return build_balanced(vals)


for vals, v, expected in examples:
    new_root = insert_brute(build_tree(vals), v)
    print(f"root = {vals}, val = {v} ->", tree_to_list(new_root))
    assert is_valid_bst(new_root)
    assert inorder(new_root) == sorted(inorder(build_tree(vals)) + [v])

root = [4, 2, 7, 1, 3], val = 5 -> [3, 1, 5, None, 2, 4, 7]
root = [40, 20, 60, 10, 30, 50, 70], val = 25 -> [30, 20, 50, 10, 25, 40, 60, None, None, None, None, None, None, None, 70]
root = [4, 2, 7, 1, 3, None, None, None, None, None, None], val = 5 -> [3, 1, 5, None, 2, 4, 7]
root = [], val = 5 -> [5]


### Độ phức tạp

- Duyệt inorder `O(n)`; `insort` `O(n)` (tìm vị trí `O(log n)` nhưng dồn phần tử `O(n)`); dựng lại cây `O(n)` ⇒ Time `O(n)`.
- Space `O(n)` cho danh sách và cây mới.

### Nút thắt

Ta **tạo lại mọi nút** (`n` nút) chỉ để thêm **một** nút. Cây BST đã chứa sẵn thông tin để **đi thẳng tới chỗ chèn** trong `O(h)`; với cây cân bằng `h ≈ log n` — nhanh hơn nhiều. Hãy đếm thử số nút phải tạo:

In [10]:
def count_nodes(root):
    n, stack = 0, [root] if root else []
    while stack:
        node = stack.pop()
        n += 1
        stack.extend(c for c in (node.left, node.right) if c)
    return n

for n in (15, 255, 4095):
    vals = list(range(0, 2 * n, 2))
    old = build_balanced(vals)
    new = insert_brute(old, 1)
    print(f"cây {n} nút: dựng lại tạo {count_nodes(new)} nút mới; chèn đúng chỗ chỉ cần tạo 1 nút và đi ~{height(old)} bước")

cây 15 nút: dựng lại tạo 16 nút mới; chèn đúng chỗ chỉ cần tạo 1 nút và đi ~4 bước
cây 255 nút: dựng lại tạo 256 nút mới; chèn đúng chỗ chỉ cần tạo 1 nút và đi ~8 bước
cây 4095 nút: dựng lại tạo 4096 nút mới; chèn đúng chỗ chỉ cần tạo 1 nút và đi ~12 bước


## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là **tận dụng thứ tự của BST**: so sánh `val` với nút hiện tại để chọn **đúng một** hướng đi. Cứ đi xuống đến khi gặp **chỗ trống** (con trỏ `None`) rồi gắn nút mới vào đó. Không cần đụng tới mọi nút khác; chỉ **một đường từ gốc xuống** bị đi qua.

Tính đúng đắn: chỗ trống đó nằm trong khoảng `(lo, hi)` mà tất cả tổ tiên trên đường đi quy định, nên gắn `val` vào đó **giữ nguyên** tính chất BST.

Từ ý tưởng này có các cách:

- **Cách 2:** đệ quy — "chèn vào cây con thích hợp rồi gán lại".
- **Cách 3:** lặp — dùng con trỏ `cur` đi xuống, dừng khi con cần đi tới trống.
- **Cách 4:** một ý tưởng **khác hẳn**: không chèn ở lá mà **chèn tại gốc** — cắt (split) cây thành hai phần "nhỏ hơn `val`" và "lớn hơn `val`", rồi cho `val` làm gốc mới với hai phần đó làm con trái/phải.

In [11]:
# minh họa nhanh: chỉ một đường từ gốc xuống bị đi qua
big = build_balanced(list(range(0, 2000, 2)))      # 1000 nút
v, cur, steps = 777, big, 0
while True:
    steps += 1
    nxt = cur.left if v < cur.val else cur.right
    if nxt is None:
        break
    cur = nxt
print(f"cây {count_nodes(big)} nút, chiều cao {height(big)}: chèn {v} chỉ đi qua {steps} nút")

cây 1000 nút, chiều cao 10: chèn 777 chỉ đi qua 10 nút


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy (chèn vào lá)

**Giải thích từng đoạn:**

1. **Cơ sở:** nếu `root is None` (cây rỗng hoặc đã đến chỗ trống) → trả về `TreeNode(val)`: nút mới chính là cây con tại chỗ đó.
2. Nếu `val < root.val`: nút mới thuộc cây con **trái**: `root.left = insert(root.left, val)`.
3. Ngược lại thuộc cây con phải: `root.right = insert(root.right, val)`.
4. Trả về `root` (gốc của cây con vừa xử lý) để nút cha **gán lại**.

**Code demo (có in vết đệ quy):**

In [12]:
def insert_recursive_demo(root, val, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}gặp chỗ trống -> tạo nút {val}")
        return TreeNode(val)
    if val < root.val:
        print(f"{pad}{val} < {root.val}: đi trái")
        root.left = insert_recursive_demo(root.left, val, depth + 1)
    else:
        print(f"{pad}{val} > {root.val}: đi phải")
        root.right = insert_recursive_demo(root.right, val, depth + 1)
    return root

t = insert_recursive_demo(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print(tree_to_list(t))

25 < 40: đi trái
  25 > 20: đi phải
    25 < 30: đi trái
      gặp chỗ trống -> tạo nút 25
[40, 20, 60, 10, 30, 50, 70, None, None, 25]


**Code nộp LeetCode (cách 2):**

In [13]:
class Solution2:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None:
            return TreeNode(val)
        if val < root.val:
            root.left = self.insertIntoBST(root.left, val)
        else:
            root.right = self.insertIntoBST(root.right, val)
        return root


for vals, v, expected in examples:
    got = tree_to_list(Solution2().insertIntoBST(build_tree(vals), v))
    assert got == expected, (got, expected)
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O(h)` với `h` là chiều cao (đi theo một đường); trường hợp xấu (cây lệch) `h = n` nên `O(n)`. Space `O(h)` cho ngăn xếp đệ quy — với cây lệch `10^4` tầng có thể **vượt giới hạn đệ quy** của Python (xem mục 13).

### 7.2. Cách 3 — Lặp (đi xuống bằng con trỏ `cur`)

**Ý tưởng khác:** không đệ quy; dùng vòng `while` đi xuống, và **dừng ngay khi con cần đi tới là `None`** (chưa rơi ra ngoài) để còn giữ `cur` là nút cha.

**Giải thích từng đoạn:**

1. Nếu `root is None` → trả `TreeNode(val)`.
2. `cur = root`. Lặp: nếu `val < cur.val`: nếu `cur.left is None` thì gắn `cur.left = TreeNode(val)` và **dừng**; ngược lại `cur = cur.left`.
3. Tương tự cho nhánh phải.
4. Trả về `root` (gốc không đổi, trừ khi cây rỗng).

**Code demo:**

In [14]:
def insert_iterative_demo(root, val):
    if root is None:
        print("cây rỗng -> nút mới là gốc")
        return TreeNode(val)
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                print(f"tại {cur.val}: {val} < {cur.val} và con trái trống -> gắn {val} làm con trái")
                cur.left = TreeNode(val)
                break
            print(f"tại {cur.val}: {val} < {cur.val} -> sang trái ({cur.left.val})")
            cur = cur.left
        else:
            if cur.right is None:
                print(f"tại {cur.val}: {val} > {cur.val} và con phải trống -> gắn {val} làm con phải")
                cur.right = TreeNode(val)
                break
            print(f"tại {cur.val}: {val} > {cur.val} -> sang phải ({cur.right.val})")
            cur = cur.right
    return root

print(tree_to_list(insert_iterative_demo(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)))
print(tree_to_list(insert_iterative_demo(None, 8)))

tại 40: 25 < 40 -> sang trái (20)
tại 20: 25 > 20 -> sang phải (30)
tại 30: 25 < 30 và con trái trống -> gắn 25 làm con trái
[40, 20, 60, 10, 30, 50, 70, None, None, 25]
cây rỗng -> nút mới là gốc
[8]


**Code nộp LeetCode (cách 3):**

In [15]:
class Solution3:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None:
            return TreeNode(val)
        cur = root
        while True:
            if val < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(val)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(val)
                    break
                cur = cur.right
        return root


for vals, v, expected in examples:
    got = tree_to_list(Solution3().insertIntoBST(build_tree(vals), v))
    assert got == expected, (got, expected)
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O(h)` (trường hợp xấu `O(n)`). Space `O(1)` — không dùng ngăn xếp đệ quy, nên **an toàn với cây lệch `10^4` nút**. Đây là cách nên nộp.

### 7.3. Cách 4 — Chèn tại gốc bằng cách cắt (split) cây

**Ý tưởng khác hẳn:** thay vì đi tới một lá rồi gắn, ta để **`val` làm gốc mới**. Muốn vậy cần hai cây BST: `L` chứa mọi giá trị **nhỏ hơn** `val`, `R` chứa mọi giá trị **lớn hơn** `val`. Ta **cắt** cây ban đầu thành `(L, R)` rồi `new_root = TreeNode(val, L, R)`.

**Cách cắt bằng một vòng lặp** (không đệ quy): giữ hai "đuôi" `lt` (cuối chuỗi bên phải của `L`) và `rt` (cuối chuỗi bên trái của `R`), với hai nút giả (dummy) làm đầu:

1. `cur = root`. Nếu `cur.val < val`: `cur` thuộc `L`; gắn nó vào `lt.right`, cập nhật `lt = cur`, rồi **đi sang phải** (`cur = cur.right`) vì cây con trái của `cur` đã hoàn toàn nằm trong `L` (đều nhỏ hơn `cur.val < val`).
2. Ngược lại `cur` thuộc `R`: gắn vào `rt.left`, `rt = cur`, rồi đi sang trái.
3. Khi `cur` hết (`None`), cắt đuôi: `lt.right = None`, `rt.left = None`.

**Code demo (có in từng bước):**

In [16]:
def split_demo(root, val):
    left_dummy, right_dummy = TreeNode(), TreeNode()
    lt, rt = left_dummy, right_dummy
    cur = root
    while cur:
        if cur.val < val:
            print(f"nút {cur.val} < {val}: thuộc L, gắn vào L rồi đi phải")
            lt.right = cur
            lt = cur
            cur = cur.right
        else:
            print(f"nút {cur.val} > {val}: thuộc R, gắn vào R rồi đi trái")
            rt.left = cur
            rt = cur
            cur = cur.left
    lt.right = None
    rt.left = None
    return left_dummy.right, right_dummy.left

L, R = split_demo(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print("L =", inorder(L), "| R =", inorder(R))
print("L hợp lệ:", is_valid_bst(L), "| R hợp lệ:", is_valid_bst(R))

nút 40 > 25: thuộc R, gắn vào R rồi đi trái
nút 20 < 25: thuộc L, gắn vào L rồi đi phải
nút 30 > 25: thuộc R, gắn vào R rồi đi trái
L = [10, 20] | R = [30, 40, 50, 60, 70]
L hợp lệ: True | R hợp lệ: True


**Code nộp LeetCode (cách 4):**

In [17]:
class Solution4:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        left_dummy, right_dummy = TreeNode(), TreeNode()
        lt, rt = left_dummy, right_dummy
        cur = root
        while cur:
            if cur.val < val:
                lt.right = cur
                lt = cur
                cur = cur.right
            else:
                rt.left = cur
                rt = cur
                cur = cur.left
        lt.right = None
        rt.left = None
        return TreeNode(val, left_dummy.right, right_dummy.left)


# nhiều đáp án đúng: kiểm tra bằng BST hợp lệ + đúng tập giá trị (không so khớp từng chữ)
for vals, v, expected in examples:
    old = build_tree(vals)
    old_vals = inorder(old)
    new_root = Solution4().insertIntoBST(old, v)
    assert is_valid_bst(new_root)
    assert inorder(new_root) == sorted(old_vals + [v])
    print(f"root = {vals}, val = {v} ->", tree_to_list(new_root))
print("Cách 4 OK")

root = [4, 2, 7, 1, 3], val = 5 -> [5, 4, 7, 2, None, None, None, 1, 3]
root = [40, 20, 60, 10, 30, 50, 70], val = 25 -> [25, 20, 40, 10, None, 30, 60, None, None, None, None, 50, 70]
root = [4, 2, 7, 1, 3, None, None, None, None, None, None], val = 5 -> [5, 4, 7, 2, None, None, None, 1, 3]
root = [], val = 5 -> [5]
Cách 4 OK


**Độ phức tạp:** Time `O(h)` (mỗi nút trên một đường được xét một lần). Space `O(1)`. Kết quả là BST hợp lệ nhưng **hình dạng khác** cách 2, 3 (nút mới nằm ở gốc). Kỹ thuật này là nền của **treap** và **splay tree**, nơi chèn tại gốc được dùng để cân bằng cây.

Vì sao `L` và `R` đều là BST? Mỗi lần gắn `cur` vào `L`, ta đã biết mọi giá trị trong cây con trái của `cur` đều `< cur.val < val` nên đi tiếp theo nhánh phải của `cur`; các nút sau này gắn vào `lt.right` lớn hơn `cur.val` (vì chúng nằm trong cây con phải của `cur`). Đối xứng với `R`.

## 8. TRACE từng bước

### 8.1. Trace cách 3 (lặp) với `root = [40,20,60,10,30,50,70]`, `val = 25`

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `bước` | thứ tự vòng lặp |
| `cur` | nút đang xét |
| `so sánh` | `val` so với `cur.val` |
| `đi` | hướng và nút kế tiếp / hành động |

In [18]:
def trace_iterative(root, val):
    print(f"chèn {val} vào {tree_to_list(root)}")
    if root is None:
        print("cây rỗng -> trả về nút mới")
        return TreeNode(val)
    cur, step = root, 0
    while True:
        step += 1
        if val < cur.val:
            nxt, side = cur.left, "trái"
        else:
            nxt, side = cur.right, "phải"
        cmp_ = "<" if val < cur.val else ">"
        if nxt is None:
            print(f"bước {step}: cur = {cur.val:>3} | {val} {cmp_} {cur.val} | con {side} trống -> GẮN {val} làm con {side} của {cur.val}")
            if side == "trái":
                cur.left = TreeNode(val)
            else:
                cur.right = TreeNode(val)
            break
        print(f"bước {step}: cur = {cur.val:>3} | {val} {cmp_} {cur.val} | đi {side} tới {nxt.val}")
        cur = nxt
    return root

r = trace_iterative(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print("kết quả:", tree_to_list(r))
r = trace_iterative(build_tree([4, 2, 7, 1, 3]), 5)
print("kết quả:", tree_to_list(r))

chèn 25 vào [40, 20, 60, 10, 30, 50, 70]
bước 1: cur =  40 | 25 < 40 | đi trái tới 20
bước 2: cur =  20 | 25 > 20 | đi phải tới 30
bước 3: cur =  30 | 25 < 30 | con trái trống -> GẮN 25 làm con trái của 30
kết quả: [40, 20, 60, 10, 30, 50, 70, None, None, 25]
chèn 5 vào [4, 2, 7, 1, 3]
bước 1: cur =   4 | 5 > 4 | đi phải tới 7
bước 2: cur =   7 | 5 < 7 | con trái trống -> GẮN 5 làm con trái của 7
kết quả: [4, 2, 7, 1, 3, 5]


Quan sát: mỗi bước loại bỏ **cả một nhánh** (khi `25 < 40` ta bỏ cả cây con phải của `40`); chỉ có `3` nút được xét trong cây `7` nút.

### 8.2. Trace cách 2 (đệ quy) — thấy việc "gán lại" khi quay về

Ta in cả lúc đi xuống lẫn lúc quay về để thấy `root.left = ...` / `root.right = ...` được thực hiện ở các nút cha:

In [19]:
def trace_recursive(root, val, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}[tới chỗ trống] tạo và trả về nút {val}")
        return TreeNode(val)
    if val < root.val:
        print(f"{pad}nút {root.val}: {val} < {root.val} -> xuống cây con trái")
        root.left = trace_recursive(root.left, val, depth + 1)
        print(f"{pad}nút {root.val}: quay về, gán con trái = {root.left.val}")
    else:
        print(f"{pad}nút {root.val}: {val} > {root.val} -> xuống cây con phải")
        root.right = trace_recursive(root.right, val, depth + 1)
        print(f"{pad}nút {root.val}: quay về, gán con phải = {root.right.val}")
    return root

r = trace_recursive(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print("kết quả:", tree_to_list(r))

nút 40: 25 < 40 -> xuống cây con trái
  nút 20: 25 > 20 -> xuống cây con phải
    nút 30: 25 < 30 -> xuống cây con trái
      [tới chỗ trống] tạo và trả về nút 25
    nút 30: quay về, gán con trái = 25
  nút 20: quay về, gán con phải = 30
nút 40: quay về, gán con trái = 20
kết quả: [40, 20, 60, 10, 30, 50, 70, None, None, 25]


Lúc quay về, các phép gán đều **ghi lại đúng giá trị cũ** ở những nút không phải nơi chèn (vì `root.left` được gán chính nó), và chỉ ở nút `30` gán thật nút mới. Đó là lý do phải **luôn gán** kết quả đệ quy.

### 8.3. Trace cách 4 (cắt và chèn tại gốc)

Trace mỗi nút thuộc `L` hay `R`, và cây `L`, `R` hình thành thế nào, với `val = 25`:

In [20]:
def trace_split(root, val):
    left_dummy, right_dummy = TreeNode(), TreeNode()
    lt, rt = left_dummy, right_dummy
    cur, step = root, 0
    while cur:
        step += 1
        if cur.val < val:
            lt.right = cur
            lt = cur
            action = f"-> L (đuôi L là {lt.val}), đi phải"
            cur = cur.right
        else:
            rt.left = cur
            rt = cur
            action = f"-> R (đuôi R là {rt.val}), đi trái"
            cur = cur.left
        print(f"bước {step}: nút {lt.val if action.startswith('-> L') else rt.val:>3} {action}")
    lt.right = None
    rt.left = None
    L, R = left_dummy.right, right_dummy.left
    print("L (inorder):", inorder(L), "| R (inorder):", inorder(R))
    new_root = TreeNode(val, L, R)
    print("cây mới (list):", tree_to_list(new_root))
    return new_root

r = trace_split(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print_tree(r)

bước 1: nút  40 -> R (đuôi R là 40), đi trái
bước 2: nút  20 -> L (đuôi L là 20), đi phải
bước 3: nút  30 -> R (đuôi R là 30), đi trái
L (inorder): [10, 20] | R (inorder): [30, 40, 50, 60, 70]
cây mới (list): [25, 20, 40, 10, None, 30, 60, None, None, None, None, 50, 70]
            70
        60
            50
    40
        30
25
    20
        10


Cây mới có gốc `25`, con trái là cây gồm `{10, 20}`, con phải là cây gồm `{30, 40, 50, 60, 70}`; hợp lệ BST dù khác cách chèn ở lá.

### 8.4. Trace cách 1 (dựng lại cây) cho cây nhỏ

In [21]:
vals = inorder(build_tree([4, 2, 7, 1, 3]))
print("inorder:", vals)
insort(vals, 5)
print("sau insort 5:", vals)
new = build_balanced(vals)
print("cây dựng lại (list):", tree_to_list(new))
print_tree(new)

inorder: [1, 2, 3, 4, 7]
sau insort 5: [1, 2, 3, 4, 5, 7]
cây dựng lại (list): [3, 1, 5, None, 2, 4, 7]
        7
    5
        4
3
        2
    1


## 9. Code nộp LeetCode

Chọn **cách 3 (lặp, chèn vào lá)**: `O(h)` thời gian, `O(1)` bộ nhớ, không đệ quy nên chịu được cây lệch `10^4` nút. Đây là code đúng class/method/signature (trên LeetCode `TreeNode` và `Optional` có sẵn; ở đây ta đã định nghĩa/import ở đầu notebook):

In [22]:
class Solution:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None:
            return TreeNode(val)
        cur = root
        while True:
            if val < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(val)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(val)
                    break
                cur = cur.right
        return root


for vals, v, expected in examples:
    got = tree_to_list(Solution().insertIntoBST(build_tree(vals), v))
    print(f"root = {vals}, val = {v} ->", got)
    assert got == expected

root = [4, 2, 7, 1, 3], val = 5 -> [4, 2, 7, 1, 3, 5]
root = [40, 20, 60, 10, 30, 50, 70], val = 25 -> [40, 20, 60, 10, 30, 50, 70, None, None, 25]
root = [4, 2, 7, 1, 3, None, None, None, None, None, None], val = 5 -> [4, 2, 7, 1, 3, 5]
root = [], val = 5 -> [5]


## 10. Vì sao thuật toán đúng

**Bất biến của vòng lặp (cách 3).** Tại mỗi thời điểm, mọi nút tổ tiên mà ta đã đi qua cho một khoảng `(lo, hi)` (cận dưới/cận trên là các tổ tiên gần nhất mà ta đã rẽ phải/trái), và `lo < val < hi`. Chứng minh quy nạp: lúc đầu `(lo, hi) = (-inf, +inf)` nên đúng. Khi ở `cur` và `val < cur.val`, ta đi trái: khoảng mới `(lo, cur.val)` chứa `val`. Tương tự khi đi phải.

**Kết thúc và vị trí gắn.** Mỗi bước đi xuống một tầng nên vòng lặp kết thúc sau tối đa `h` bước, khi con cần đi tới là `None`. Khoảng `(lo, hi)` lúc đó cũng là khoảng hợp lệ cho **mọi** giá trị trong cây con còn trống đó. Gắn `val` ở đó:

- `val` nằm trong `(lo, hi)`, tức tuân thủ ràng buộc của **mọi tổ tiên** (nhỏ hơn các tổ tiên mà ta rẽ trái, lớn hơn các tổ tiên mà ta rẽ phải);
- vì `val` là lá mới, nó không có cây con nên không ràng buộc gì thêm.

Mọi nút khác không đổi nên vẫn thỏa BST ⇒ cây mới là BST và chứa đúng các giá trị cũ cùng `val`. Vì `val` chưa có trong cây nên không bao giờ gặp trường hợp "bằng" (nhánh `else` chỉ xảy ra khi `val > cur.val`).

**Cách 2** giống hệt nhưng viết bằng đệ quy: `insert(root, val)` trả về gốc của cây con sau khi chèn; việc gán `root.left/right = ...` ghi lại (hoặc giữ nguyên) con trỏ; điều kiện cơ sở `root is None` tạo nút mới.

**Cách 4.** `L` chỉ gồm các nút `< val`, `R` chỉ gồm các nút `> val`, và cả hai vẫn là BST (chứng minh ở mục 7.3). Nút mới `val` lớn hơn mọi giá trị trong `L` và nhỏ hơn mọi giá trị trong `R`, nên `TreeNode(val, L, R)` là BST. Mọi nút cũ xuất hiện đúng một lần (mỗi nút được gắn vào đúng một trong hai cây).

**Cách 1.** Inorder của BST là dãy tăng; chèn `val` đúng chỗ giữ dãy tăng; dựng cây cân bằng từ dãy tăng cho BST hợp lệ (phần tử giữa lớn hơn nửa trái và nhỏ hơn nửa phải).

**Cây rỗng.** Cách 1: danh sách `[val]` → cây một nút; cách 2/3: trả `TreeNode(val)`; cách 4: vòng lặp không chạy, `L = R = None` → `TreeNode(val)`.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Inorder + `insort` + dựng lại | `O(n)` | `O(n)` |
| 2. Đệ quy chèn vào lá | `O(h)` (xấu nhất `O(n)`) | `O(h)` ngăn xếp |
| 3. Lặp chèn vào lá (nộp) | `O(h)` (xấu nhất `O(n)`) | `O(1)` |
| 4. Cắt và chèn tại gốc | `O(h)` (xấu nhất `O(n)`) | `O(1)` |

`h` là chiều cao cây: `O(log n)` nếu cân bằng, `O(n)` nếu lệch thành chuỗi. Đo thực tế số bước (độ sâu chỗ chèn) trên cây cân bằng và cây lệch:

In [23]:
def insert_depth(root, val):
    """Số nút đi qua khi chèn val vào lá (dùng để minh họa O(h))."""
    cur, steps = root, 0
    while cur:
        steps += 1
        cur = cur.left if val < cur.val else cur.right
    return steps

for n in (127, 1023, 8191):
    balanced = build_balanced(list(range(0, 2 * n, 2)))
    print(f"cân bằng {n:>5} nút: h = {height(balanced):>2}, chèn đi qua {insert_depth(balanced, 1):>2} nút")
for n in (100, 1000, 10000):
    chain = chain_tree(n)
    print(f"lệch (chuỗi) {n:>5} nút: chèn một giá trị lớn đi qua {insert_depth(chain, n + 1):>5} nút")

cân bằng   127 nút: h =  7, chèn đi qua  7 nút
cân bằng  1023 nút: h = 10, chèn đi qua 10 nút
cân bằng  8191 nút: h = 13, chèn đi qua 13 nút
lệch (chuỗi)   100 nút: chèn một giá trị lớn đi qua   100 nút
lệch (chuỗi)  1000 nút: chèn một giá trị lớn đi qua  1000 nút
lệch (chuỗi) 10000 nút: chèn một giá trị lớn đi qua 10000 nút


Cây cân bằng cho `log n` bước; cây lệch cho `n` bước. LeetCode không đảm bảo cây cân bằng nên ta tính theo `O(h)` và hiểu xấu nhất là `O(n)`.

Thời gian thực tế (không in số chính xác vì phụ thuộc máy) cho thấy cách 3 nhanh hơn rõ rệt so với dựng lại cây:

In [24]:
base = build_balanced(list(range(0, 20000, 2)))     # 10^4 nút
vals_all = inorder(base)

def timed(f, repeat=30):
    t0 = time.perf_counter()
    for _ in range(repeat):
        f()
    return (time.perf_counter() - t0) / repeat * 1000

t_brute = timed(lambda: insert_brute(base, 1))          # base không bị sửa

grow = build_balanced(list(range(0, 20000, 2)))
odd_values = iter(range(1, 20000, 2))                     # mỗi lần chèn một số lẻ chưa có trong cây
t_leaf = timed(lambda: Solution().insertIntoBST(grow, next(odd_values)), 200)
print(f"brute (dựng lại 10^4 nút): ~{t_brute:7.2f} ms mỗi lần")
print(f"chèn vào lá (cách 3) trên cây ~10^4 nút: ~{t_leaf:7.4f} ms mỗi lần")
print("-> chèn tại chỗ chỉ đi O(h) bước, rẻ hơn dựng lại O(n) rất nhiều khi n lớn")

brute (dựng lại 10^4 nút): ~   5.40 ms mỗi lần
chèn vào lá (cách 3) trên cây ~10^4 nút: ~ 0.0010 ms mỗi lần
-> chèn tại chỗ chỉ đi O(h) bước, rẻ hơn dựng lại O(n) rất nhiều khi n lớn


## 12. Bộ test

### 12.1. Hàm kiểm tra đáp án (vì có nhiều đáp án đúng)

Vì đề cho phép **nhiều BST hợp lệ**, ta không so sánh với một cây cố định. Thay vào đó định nghĩa `check_result(old_values, val, new_root)` kiểm tra: (1) cây là **BST hợp lệ**; (2) **tập giá trị** (inorder) đúng bằng `old_values ∪ {val}`, không mất, không thừa, không trùng; (3) số nút tăng đúng 1.

In [25]:
def check_result(old_values, val, new_root):
    assert is_valid_bst(new_root), "không còn là BST"
    got = inorder(new_root)
    assert got == sorted(old_values + [val]), (got, sorted(old_values + [val]))
    assert count_nodes(new_root) == len(old_values) + 1
    return True


solvers = [
    ("cách 1", insert_brute),
    ("cách 2", Solution2().insertIntoBST),
    ("cách 3", Solution3().insertIntoBST),
    ("cách 4", Solution4().insertIntoBST),
]


def check_all(vals_list, val, exact=None):
    """vals_list: list kiểu LeetCode của BST gốc. exact: đáp án 'đúng từng chữ' cho cách 2, 3 (nếu có)."""
    old_values = inorder(build_tree(vals_list))
    for name, f in solvers:
        new_root = f(build_tree(vals_list), val)
        check_result(old_values, val, new_root)
    if exact is not None:
        for f in (Solution2().insertIntoBST, Solution3().insertIntoBST, Solution().insertIntoBST):
            assert tree_to_list(f(build_tree(vals_list), val)) == exact, (vals_list, val)


for vals, v, expected in examples:
    check_all(vals, v, expected)
print("ví dụ đề: OK (mọi cách hợp lệ; cách 2, 3 và cách nộp còn khớp đúng từng chữ đáp án mẫu)")

ví dụ đề: OK (mọi cách hợp lệ; cách 2, 3 và cách nộp còn khớp đúng từng chữ đáp án mẫu)


### 12.2. Edge case

Các trường hợp: cây rỗng, cây một nút (chèn nhỏ hơn / lớn hơn), chèn nhỏ nhất, chèn lớn nhất, chèn ở giữa, giá trị âm, cây lệch trái, cây lệch phải, biên giá trị `±10^8`.

In [26]:
edge = [
    ([], 0),
    ([], -10**8),
    ([5], 3),
    ([5], 8),
    ([5, 3], 4),                  # chèn làm con phải của lá trái
    ([5, None, 8], 6),            # chèn làm con trái của lá phải
    ([10, 5, 15, 3, 7, 12, 20], 1),       # nhỏ nhất
    ([10, 5, 15, 3, 7, 12, 20], 100),     # lớn nhất
    ([10, 5, 15, 3, 7, 12, 20], 8),       # ở giữa, sâu
    ([0, -5, 5], -3),
    ([10**8, -10**8], 0),
    ([3, 2, None, 1], 0),         # cây lệch trái, chèn nhỏ nhất
    ([1, None, 2, None, 3], 4),   # cây lệch phải, chèn lớn nhất
    ([2, None, 3], 1),
]
for vals, v in edge:
    check_all(vals, v)
print("edge case: OK")

# các kết quả cụ thể của cách nộp (nhánh 'chèn vào lá') cho vài edge case
assert tree_to_list(Solution().insertIntoBST(None, 7)) == [7]
assert tree_to_list(Solution().insertIntoBST(build_tree([5]), 3)) == [5, 3]
assert tree_to_list(Solution().insertIntoBST(build_tree([5]), 8)) == [5, None, 8]
assert tree_to_list(Solution().insertIntoBST(build_tree([5, None, 8]), 6)) == [5, None, 8, 6]
print("kết quả cụ thể của cách nộp: OK")

edge case: OK
kết quả cụ thể của cách nộp: OK


### 12.3. Phản ví dụ cho các cách chèn sai

Ta viết các cách chèn sai rồi dùng **bộ kiểm tra** để bắt chúng:

In [27]:
class WrongSwapDirection:
    """SAI: đi phải khi val < cur.val (ngược chiều)."""
    def insertIntoBST(self, root, val):
        if root is None:
            return TreeNode(val)
        if val < root.val:
            root.right = self.insertIntoBST(root.right, val)
        else:
            root.left = self.insertIntoBST(root.left, val)
        return root


class WrongAlwaysRoot:
    """SAI: nối nút mới làm con của gốc, không cần biết có sẵn con hay chưa (đè mất cây con)."""
    def insertIntoBST(self, root, val):
        if root is None:
            return TreeNode(val)
        if val < root.val:
            root.left = TreeNode(val)
        else:
            root.right = TreeNode(val)
        return root


vals, v = [40, 20, 60, 10, 30, 50, 70], 25
old_values = inorder(build_tree(vals))
for cls in (WrongSwapDirection, WrongAlwaysRoot):
    new_root = cls().insertIntoBST(build_tree(vals), v)
    try:
        check_result(old_values, v, new_root)
        print(cls.__name__, ": (không bắt được!)")
    except AssertionError as ex:
        print(f"{cls.__name__:20}: bị bắt bởi bộ kiểm tra ({str(ex)[:60]})")

WrongSwapDirection  : bị bắt bởi bộ kiểm tra (không còn là BST)
WrongAlwaysRoot     : bị bắt bởi bộ kiểm tra (([25, 40, 50, 60, 70], [10, 20, 25, 30, 40, 50, 60, 70]))


- `WrongSwapDirection` chèn sai chiều nên `25 < 40` lại rơi vào nhánh **phải** của `40`: cây không còn là BST.
- `WrongAlwaysRoot` đè mất cả cây con `20` (và `10`, `30`) nên **mất giá trị**: tập giá trị không còn đủ.

Một lý do vì sao ta kiểm tra bằng **BST hợp lệ + tập giá trị** thay vì chỉ so với "đáp án mẫu": cả hai lỗi trên có thể vô tình khớp mẫu ở những input đơn giản, còn bộ kiểm tra tổng quát bắt được.

### 12.4. Kiểm tra ngẫu nhiên (seed cố định)

Sinh BST ngẫu nhiên **đúng ràng buộc** (giá trị khác nhau trong `[-10^8, 10^8]`, `0` đến vài chục nút) bằng cách chèn các giá trị phân biệt theo thứ tự ngẫu nhiên; chọn `val` ngẫu nhiên chưa có trong cây. Kiểm tra mọi cách bằng `check_result`, và với cách 2, 3 còn kiểm tra hai cách cho **cùng cây** (đệ quy và lặp cùng chèn vào lá).

In [28]:
def random_bst_values(rng, n, lo=-10**8, hi=10**8):
    return rng.sample(range(lo, hi + 1), n) if hi - lo > 10**6 else rng.sample(range(lo, hi + 1), min(n, hi - lo + 1))


def make_bst(values):
    root = None
    for v in values:                       # chèn lần lượt bằng cách nộp (đã kiểm chứng ở trên)
        root = Solution().insertIntoBST(root, v)
    return root


rng = random.Random(20701)
cases = 0
for _ in range(1500):
    n = rng.randint(0, 40)
    values = random_bst_values(rng, n + 1)
    val, base_values = values[0], values[1:]
    root = make_bst(base_values)
    assert is_valid_bst(root)
    old_values = inorder(root)
    results = {}
    for name, f in solvers:
        new_root = f(make_bst(base_values), val)      # mỗi cách làm trên một bản sao riêng
        check_result(old_values, val, new_root)
        results[name] = tree_to_list(new_root)
    assert results["cách 2"] == results["cách 3"]
    cases += 1
print(f"{cases} cây ngẫu nhiên: mọi cách cho BST hợp lệ đúng tập giá trị; cách 2 và 3 cho cùng hình dạng")

# giá trị trong dải nhỏ để tạo nhiều chỗ chèn sát nhau, cây thường lệch
for _ in range(500):
    n = rng.randint(0, 12)
    values = rng.sample(range(-15, 16), n + 1)
    val, base_values = values[0], values[1:]
    root = make_bst(base_values)
    check_all(tree_to_list(root), val)
print("500 cây dải nhỏ: OK")

1500 cây ngẫu nhiên: mọi cách cho BST hợp lệ đúng tập giá trị; cách 2 và 3 cho cùng hình dạng
500 cây dải nhỏ: OK


In [29]:
# cây lệch lớn: 10^4 nút -> cách lặp phải chịu được, đệ quy cần nới giới hạn
chain = chain_tree(10**4)
res = Solution().insertIntoBST(chain, 10**4 + 1)
assert is_valid_bst(res) and count_nodes(res) == 10**4 + 1
print("cách lặp trên chuỗi 10^4 nút: OK")

chain = chain_tree(10**4)
try:
    Solution2().insertIntoBST(chain, 10**4 + 1)
    print("cách đệ quy trên chuỗi 10^4 nút: không lỗi")
except RecursionError:
    print("cách đệ quy trên chuỗi 10^4 nút: RecursionError (giới hạn mặc định)")

# nếu nới giới hạn thì cách đệ quy cũng chạy được (trên LeetCode giới hạn đã được nới sẵn)
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
chain = chain_tree(10**4)
res2 = Solution2().insertIntoBST(chain, 10**4 + 1)
sys.setrecursionlimit(old_limit)
assert is_valid_bst(res2) and count_nodes(res2) == 10**4 + 1
print("cách đệ quy với giới hạn nới: OK")

# cách 4 (cắt) trên chuỗi: vẫn O(h) lặp
chain = chain_tree(10**4)
res4 = Solution4().insertIntoBST(chain, 10**4 + 1)
assert is_valid_bst(res4) and count_nodes(res4) == 10**4 + 1
print("cách cắt trên chuỗi 10^4 nút: OK")

cách lặp trên chuỗi 10^4 nút: OK
cách đệ quy trên chuỗi 10^4 nút: RecursionError (giới hạn mặc định)
cách đệ quy với giới hạn nới: OK
cách cắt trên chuỗi 10^4 nút: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên xử lý cây rỗng

**Code sai:**

In [30]:
class Wrong1:
    def insertIntoBST(self, root, val):
        cur = root
        while True:
            if val < cur.val:               # root = None -> cur = None -> AttributeError
                if cur.left is None:
                    cur.left = TreeNode(val)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(val)
                    break
                cur = cur.right
        return root

try:
    Wrong1().insertIntoBST(None, 5)
except AttributeError as ex:
    print("lỗi:", ex)
print("đúng:", tree_to_list(Solution().insertIntoBST(None, 5)))

lỗi: 'NoneType' object has no attribute 'val'
đúng: [5]


**Vì sao sai:** ràng buộc cho phép `0` nút. Khi `root is None`, kết quả phải là cây **một nút `val`** và **gốc thay đổi**, nên cần `if root is None: return TreeNode(val)` ở đầu.

### Lỗi 2: Gọi đệ quy mà không gán lại kết quả

In [31]:
class Wrong2:
    def insertIntoBST(self, root, val):
        if root is None:
            return TreeNode(val)
        if val < root.val:
            self.insertIntoBST(root.left, val)       # SAI: bỏ kết quả trả về
        else:
            self.insertIntoBST(root.right, val)
        return root

for vals, v in (([4, 2, 7, 1, 3], 5), ([4, 2, 7, 1, 3, None, 8], 6), ([], 5)):
    got = tree_to_list(Wrong2().insertIntoBST(build_tree(vals), v))
    print(f"root = {vals}, val = {v}: sai -> {got}")
print("đúng:", tree_to_list(Solution2().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 5)))

root = [4, 2, 7, 1, 3], val = 5: sai -> [4, 2, 7, 1, 3]
root = [4, 2, 7, 1, 3, None, 8], val = 6: sai -> [4, 2, 7, 1, 3, None, 8]
root = [], val = 5: sai -> [5]
đúng: [4, 2, 7, 1, 3, 5]


**Vì sao sai:** khi tới chỗ trống, hàm **tạo** nút mới và **trả về** nó; nếu nút cha không gán `root.left = ...` thì nút mới **bị bỏ mất** (nút cha vẫn trỏ `None`). Với cây rỗng cũng vậy (nhưng lúc đó hàm trả về nút mới đúng, nên kết quả vẫn đúng ở đầu vào rỗng).

### Lỗi 3: Đè mất cây con khi chỉ nhìn một mức
(Đã minh họa ở 12.3 với `WrongAlwaysRoot`.) Viết `root.left = TreeNode(val)` mà **không kiểm tra** `root.left` đã có cây con chưa sẽ **ghi đè** cả cây con đó.

In [32]:
t = build_tree([40, 20, 60, 10, 30, 50, 70])
wrong = WrongAlwaysRoot().insertIntoBST(t, 25)
print("sai   :", tree_to_list(wrong), "| số nút còn lại =", count_nodes(wrong), "(mất 10, 20, 30!)")
right = Solution().insertIntoBST(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
print("đúng  :", tree_to_list(right), "| số nút =", count_nodes(right))

sai   : [40, 25, 60, None, None, 50, 70] | số nút còn lại = 5 (mất 10, 20, 30!)
đúng  : [40, 20, 60, 10, 30, 50, 70, None, None, 25] | số nút = 8


**Cách sửa:** nếu con tương ứng **đã có** thì đi xuống tiếp (đệ quy hoặc `cur = cur.left`); chỉ khi nó là `None` mới tạo nút mới.

### Lỗi 4: Để `cur` rơi ra `None` rồi mất nút cha

In [33]:
class Wrong4:
    def insertIntoBST(self, root, val):
        if root is None:
            return TreeNode(val)
        cur = root
        while cur:                                  # khi thoát vòng lặp, cur = None
            cur = cur.left if val < cur.val else cur.right
        cur = TreeNode(val)                         # SAI: chỉ gán biến cục bộ, không gắn vào cây
        return root

got = Wrong4().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 5)
print("sai  :", tree_to_list(got), "(giá trị 5 biến mất)")
print("đúng :", tree_to_list(Solution().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 5)))

sai  : [4, 2, 7, 1, 3] (giá trị 5 biến mất)
đúng : [4, 2, 7, 1, 3, 5]


**Vì sao sai:** gán `cur = TreeNode(val)` chỉ làm biến `cur` trỏ sang nút mới, **không** sửa con trỏ `left/right` của nút cha. Phải dừng khi **con cần đi tới là `None`** và gán `cur.left = ...` hoặc `cur.right = ...` trên nút cha.

### Lỗi 5: So sánh sai chiều
Đã minh họa với `WrongSwapDirection` ở mục 12.3: cây sau khi chèn không còn là BST.

In [34]:
vals, v = [4, 2, 7, 1, 3], 5
bad = WrongSwapDirection().insertIntoBST(build_tree(vals), v)
print("kết quả sai:", tree_to_list(bad), "| BST hợp lệ?", is_valid_bst(bad))
print("kết quả đúng:", tree_to_list(Solution().insertIntoBST(build_tree(vals), v)), "| BST hợp lệ?", True)

kết quả sai: [4, 2, 7, 1, 3, None, None, 5] | BST hợp lệ? False
kết quả đúng: [4, 2, 7, 1, 3, 5] | BST hợp lệ? True


**Vì sao sai:** BST yêu cầu **nhỏ hơn đi trái, lớn hơn đi phải**; đảo chiều sẽ gắn giá trị vào phần cây mà tổ tiên cấm.

### Lỗi 6: Đệ quy trên cây lệch `10^4` nút
Python mặc định chỉ cho ~1000 tầng đệ quy. Với cây lệch, `Solution2` gây `RecursionError` (đã thấy ở mục 12.4). **Giải pháp:** dùng cách lặp (cách 3) khi nộp, hoặc nới `sys.setrecursionlimit`.

In [35]:
chain = chain_tree(3000)
try:
    Solution2().insertIntoBST(chain, 3001)
    print("không lỗi")
except RecursionError:
    print("Solution2 (đệ quy) trên chuỗi 3000 nút -> RecursionError")
res = Solution().insertIntoBST(chain_tree(3000), 3001)
print("Solution (lặp) trên cùng cây: OK, số nút =", count_nodes(res))

Solution2 (đệ quy) trên chuỗi 3000 nút -> RecursionError
Solution (lặp) trên cùng cây: OK, số nút = 3001


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** đề cho **BST** và yêu cầu chèn/tìm/xoá một giá trị → **đi một đường từ gốc**, so sánh để chọn trái/phải. Thời gian `O(h)` chứ không phải `O(n)`.
- **BST ⇒ inorder tăng dần**: dùng để kiểm tra/ghi nhớ; mọi bài "kiểm tra BST", "kth smallest trong BST", "khôi phục BST" đều quay về tính chất này.
- **Hàm đệ quy trả về gốc cây con** (`node.left = f(node.left)`) là mẫu rất phổ biến khi **sửa cây**: chèn, xoá, cắt tỉa. Nhớ **luôn gán** kết quả trả về.
- **Cây rỗng** là một trường hợp phải tự thử: gốc thay đổi nên bắt buộc phải **trả về** gốc.
- **Nhiều đáp án đúng ⇒ test bằng hàm kiểm tra tính hợp lệ**: ở đây là "BST hợp lệ + đúng tập giá trị".
- **Chèn vào lá** đơn giản nhất và không đổi hình dạng phần cây cũ; **chèn tại gốc** (split) là nền của treap/splay.
- **Chiều cao**: cây lệch làm `O(h)` biến thành `O(n)` và làm đệ quy tràn; BST tự cân bằng (AVL, đỏ-đen) tránh điều này.
- **Python:** `while True` + `break`, và `cur.left is None` để nhìn trước một bước thay vì rơi vào `None`.

## 15. Biến thể

1. **Tìm kiếm trong BST (LeetCode 700):** cùng đường đi nhưng chỉ cần trả về nút có giá trị `val` (hoặc `None`).
2. **Xoá một nút khỏi BST (LeetCode 450):** nút lá, một con, hai con (thay bằng nút kế nhiệm inorder).
3. **Kiểm tra BST hợp lệ (LeetCode 98):** dùng khoảng `(lo, hi)` như `is_valid_bst` ở trên.
4. **Dựng BST từ duyệt tiền tự (LeetCode 1008)** và **từ mảng đã sắp xếp (LeetCode 108)**.
5. **Chèn vào BST cho phép trùng:** quy ước giá trị bằng đi trái (hoặc phải) hoặc đếm số lần.
6. **Cây tự cân bằng:** AVL/đỏ-đen/treap chèn có xoay để giữ `h = O(log n)`.
7. **Chèn nhiều giá trị** và tính chiều cao kết quả: thứ tự chèn quyết định hình dạng cây.

Demo biến thể 1 (tìm kiếm) và 7 (thứ tự chèn quyết định chiều cao):

In [36]:
def search_bst(root, val):
    cur = root
    while cur and cur.val != val:
        cur = cur.left if val < cur.val else cur.right
    return cur


def insert_all(values):
    root = None
    for v in values:
        root = Solution().insertIntoBST(root, v)
    return root


t = insert_all([40, 20, 60, 10, 30, 50, 70, 25])
print("tìm 25:", search_bst(t, 25).val, "| tìm 26:", search_bst(t, 26))

n = 63
sorted_order = list(range(1, n + 1))
def median_first(lo, hi):
    if lo > hi:
        return []
    mid = (lo + hi) // 2
    return [mid] + median_first(lo, mid - 1) + median_first(mid + 1, hi)

print("chèn 1..63 theo thứ tự tăng:  chiều cao =", height(insert_all(sorted_order)))
print("chèn 63 giá trị theo thứ tự trung vị trước: chiều cao =", height(insert_all(median_first(1, n))))
rng = random.Random(15)
shuffled = sorted_order[:]
rng.shuffle(shuffled)
print("chèn theo thứ tự ngẫu nhiên:   chiều cao =", height(insert_all(shuffled)))
assert height(insert_all(sorted_order)) == 63
assert height(insert_all(median_first(1, n))) == 6

tìm 25: 25 | tìm 26: None
chèn 1..63 theo thứ tự tăng:  chiều cao = 63
chèn 63 giá trị theo thứ tự trung vị trước: chiều cao = 6
chèn theo thứ tự ngẫu nhiên:   chiều cao = 13


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Inorder + `insort` + dựng lại | Bỏ qua cấu trúc cây, dựng cây cân bằng mới | `O(n)` | `O(n)` | Chỉ để kiểm chứng; hữu ích nếu cần cây cân bằng |
| 2. Đệ quy chèn vào lá | `root.left/right = insert(...)` | `O(h)` | `O(h)` | Ngắn gọn; cẩn thận cây lệch sâu |
| 3. Lặp chèn vào lá | Con trỏ `cur`, dừng khi con trống | `O(h)` | `O(1)` | **Cách nộp**: nhanh, an toàn |
| 4. Cắt (split) và chèn tại gốc | `L < val < R`, `val` làm gốc mới | `O(h)` | `O(1)` | Nền của treap/splay; khi muốn nút mới ở gốc |

Khác nhau thật về ý tưởng: cách 1 phá cấu trúc và dựng lại; cách 2, 3 cùng ý tưởng "chèn vào lá" (đệ quy vs lặp, cho **cùng một cây**); cách 4 cho một cây **hình dạng khác** vì nút mới ở gốc.

In [37]:
# so sánh hình dạng kết quả trên cùng input
vals, v = [40, 20, 60, 10, 30, 50, 70], 25
for name, f in solvers:
    r = f(build_tree(vals), v)
    print(f"{name}: list = {tree_to_list(r)} | chiều cao = {height(r)}")

# số nút bị 'đụng' tới (đổi con trỏ) — minh họa khác biệt giữa các cách
def count_changed_pointers(before_root, f, val):
    snapshot = {}
    stack = [before_root] if before_root else []
    while stack:
        nd = stack.pop()
        snapshot[id(nd)] = (id(nd.left) if nd.left else None, id(nd.right) if nd.right else None)
        stack.extend(c for c in (nd.left, nd.right) if c)
    after = f(before_root, val)
    changed = 0
    stack = [after] if after else []
    while stack:
        nd = stack.pop()
        if id(nd) in snapshot:
            now = (id(nd.left) if nd.left else None, id(nd.right) if nd.right else None)
            changed += now != snapshot[id(nd)]
        stack.extend(c for c in (nd.left, nd.right) if c)
    return changed

for name, f in solvers[1:]:
    print(f"{name}: số nút cũ bị đổi con trỏ = {count_changed_pointers(build_tree(vals), f, v)}")

cách 1: list = [30, 20, 50, 10, 25, 40, 60, None, None, None, None, None, None, None, 70] | chiều cao = 4
cách 2: list = [40, 20, 60, 10, 30, 50, 70, None, None, 25] | chiều cao = 4
cách 3: list = [40, 20, 60, 10, 30, 50, 70, None, None, 25] | chiều cao = 4
cách 4: list = [25, 20, 40, 10, None, 30, 60, None, None, None, None, 50, 70] | chiều cao = 4
cách 2: số nút cũ bị đổi con trỏ = 1
cách 3: số nút cũ bị đổi con trỏ = 1
cách 4: số nút cũ bị đổi con trỏ = 2


Cách 2, 3 chỉ đổi **một** con trỏ (của nút cha chỗ chèn); cách 4 đổi nhiều con trỏ hơn vì phải cắt cây ra rồi ghép lại quanh gốc mới.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [38]:
print(tree_to_list(Solution().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 5)))
print(tree_to_list(Solution().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 0)))
print(tree_to_list(Solution().insertIntoBST(build_tree([]), 9)))

[4, 2, 7, 1, 3, 5]
[4, 2, 7, 1, 3, None, None, 0]
[9]


**Bài 2 (tính tay).** Chèn lần lượt các giá trị `8, 3, 10, 1, 6, 14, 4` vào cây rỗng (theo cách chèn vào lá). Vẽ cây, rồi viết dạng list kiểu LeetCode vào `ans2` (đang là `None`) và chiều cao vào `ans2_h`.

In [39]:
ans2 = None      # TODO: list kiểu LeetCode của cây sau 7 lần chèn
ans2_h = None    # TODO: chiều cao cây

**Bài 3 (điền TODO).** Hoàn thành `bst_contains(root, val)` trả về `True` nếu `val` có trong BST (lặp, `O(h)`).

In [40]:
def bst_contains(root, val):
    # TODO: đi từ gốc, so sánh để chọn trái/phải, dừng khi cur là None hoặc tìm thấy
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_insert(root, val)` theo cách **lặp** (chèn vào lá).

In [41]:
def my_insert(root, val):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `build_min_height(values)`: nhận list giá trị **phân biệt** (thứ tự bất kỳ) và trả về BST có chiều cao **nhỏ nhất** có thể (gợi ý: sắp xếp rồi chèn theo "thứ tự trung vị trước", hoặc dựng thẳng như `build_balanced`). Chiều cao phải bằng `ceil(log2(n + 1))`. (b) Viết `insert_with_depth(root, val)` trả về cặp `(root_mới, độ_sâu_của_nút_mới)` với độ sâu của gốc là `1`.

In [42]:
def build_min_height(values):
    # TODO
    pass

def insert_with_depth(root, val):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: với mỗi trường hợp, đi từ gốc so sánh để biết chèn ở đâu.
- Bài 2: chèn `8` làm gốc; `3 < 8` trái; `10 > 8` phải; ... cứ so sánh từ gốc.
- Bài 3: giống đường đi khi chèn nhưng chỉ cần tìm.
- Bài 4: `while True`, nhìn trước con kế tiếp có `None` không.
- Bài 5: (a) sắp xếp rồi lấy phần tử giữa làm gốc; (b) đếm số nút đi qua.

**Mức 2 (vừa):**
- Bài 2: gốc `8`; trái: `3` (con trái `1`, con phải `6` có con trái `4`); phải: `10` (con phải `14`). Chiều cao = `4` (đường `8 → 3 → 6 → 4`).
- Bài 3: `while cur and cur.val != val: cur = cur.left if val < cur.val else cur.right`; `return cur is not None`.
- Bài 4: nhánh `val < cur.val` và nhánh còn lại; trong mỗi nhánh nếu con là `None` thì gắn và `break`.
- Bài 5: (a) `sorted_vals = sorted(values)`, dựng đệ quy trên đoạn `[lo, hi]`; (b) trong vòng lặp tăng biến `depth` mỗi lần đi xuống, bắt đầu từ `1` ở gốc.

**Mức 3 (gần đáp án):**
- Bài 3: xem `search_bst` ở mục 15.
- Bài 4: xem mục 9, code nộp.
- Bài 5: (a) dùng `build_balanced(sorted(values))`; (b) nút mới có độ sâu bằng (số nút trên đường đi) + 1; với cây rỗng độ sâu là 1.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [43]:
# Bài 1
assert tree_to_list(Solution().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 5)) == [4, 2, 7, 1, 3, 5]
assert tree_to_list(Solution().insertIntoBST(build_tree([4, 2, 7, 1, 3]), 0)) == [4, 2, 7, 1, 3, None, None, 0]
assert tree_to_list(Solution().insertIntoBST(build_tree([]), 9)) == [9]

# Bài 2
ans2 = [8, 3, 10, 1, 6, None, 14, None, None, 4]
ans2_h = 4
built = insert_all([8, 3, 10, 1, 6, 14, 4])
assert ans2 == tree_to_list(built)
assert ans2_h == height(built)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [44]:
# Bài 3
def bst_contains(root, val):
    cur = root
    while cur and cur.val != val:
        cur = cur.left if val < cur.val else cur.right
    return cur is not None

t = insert_all([8, 3, 10, 1, 6, 14, 4])
assert all(bst_contains(t, x) for x in (8, 3, 10, 1, 6, 14, 4))
assert not any(bst_contains(t, x) for x in (0, 2, 5, 7, 9, 100))
assert bst_contains(None, 1) is False
print("Bài 3 OK")

Bài 3 OK


In [45]:
# Bài 4
def my_insert(root, val):
    if root is None:
        return TreeNode(val)
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = TreeNode(val)
                break
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = TreeNode(val)
                break
            cur = cur.right
    return root

rng = random.Random(4)
for _ in range(300):
    values = rng.sample(range(-100, 101), rng.randint(1, 30))
    a, b = None, None
    for x in values:
        a = my_insert(a, x)
        b = Solution().insertIntoBST(b, x)
    assert tree_to_list(a) == tree_to_list(b)
    assert is_valid_bst(a) and inorder(a) == sorted(values)
print("Bài 4 OK")

Bài 4 OK


In [46]:
# Bài 5
def build_min_height(values):
    return build_balanced(sorted(values))


def insert_with_depth(root, val):
    if root is None:
        return TreeNode(val), 1
    cur, depth = root, 1
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = TreeNode(val)
                return root, depth + 1
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = TreeNode(val)
                return root, depth + 1
            cur = cur.right
        depth += 1


import math
rng = random.Random(5)
for n in range(1, 130):
    values = rng.sample(range(-1000, 1000), n)
    t = build_min_height(values)
    assert is_valid_bst(t) and inorder(t) == sorted(values)
    assert height(t) == math.ceil(math.log2(n + 1)), n

r, d = insert_with_depth(None, 5)
assert tree_to_list(r) == [5] and d == 1
r, d = insert_with_depth(build_tree([40, 20, 60, 10, 30, 50, 70]), 25)
assert d == 4 and tree_to_list(r) == [40, 20, 60, 10, 30, 50, 70, None, None, 25]
r, d = insert_with_depth(build_tree([4, 2, 7, 1, 3]), 5)
assert d == 3
print("Bài 5 OK")

Bài 5 OK

## 20. Template nhớ nhanh + Checklist trước khi submit

**Template lặp (nộp):**

```python
class Solution:
    def insertIntoBST(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None:
            return TreeNode(val)
        cur = root
        while True:
            if val < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(val)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(val)
                    break
                cur = cur.right
        return root
```

**Template đệ quy:**

```python
def insertIntoBST(self, root, val):
    if root is None:
        return TreeNode(val)
    if val < root.val:
        root.left = self.insertIntoBST(root.left, val)
    else:
        root.right = self.insertIntoBST(root.right, val)
    return root
```

**Checklist trước khi submit:**

- [ ] Cây rỗng (`root is None`) -> trả `TreeNode(val)`?
- [ ] Nhỏ hơn đi **trái**, lớn hơn đi **phải**?
- [ ] Đệ quy: có **gán lại** `root.left = ...` / `root.right = ...` và `return root`?
- [ ] Lặp: dừng khi **con cần đi tới là `None`** (không để `cur` rơi ra `None`)?
- [ ] Không đè mất cây con có sẵn (chỉ gắn khi chỗ đó là `None`)?
- [ ] Cây lệch `10^4` nút: cách lặp an toàn; đệ quy cần cẩn thận giới hạn?
- [ ] Test bằng: BST hợp lệ + đúng tập giá trị (vì có nhiều đáp án), kèm cây rỗng, một nút, lệch trái/phải?